<a href="https://colab.research.google.com/github/Seif357/Expert-Packet-Analyzer/blob/main/Expert-Packet-Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

SUSPICIOUS_PORTS = {
    21:    "FTP — sends passwords unencrypted",
    23:    "Telnet — fully unencrypted remote access",
    25:    "SMTP — often abused for spam/relay",
    69:    "TFTP — no authentication at all",
    135:   "RPC — exploited by many Windows worms",
    137:   "NetBIOS — Windows file-share, often exploited",
    139:   "NetBIOS — Windows file-share, often exploited",
    445:   "SMB — WannaCry and most Windows worms use this",
    1433:  "MSSQL — database, common brute-force target",
    3306:  "MySQL — database, common brute-force target",
    3389:  "RDP — remote desktop, constantly brute-forced",
    4444:  "Metasploit default shell / reverse shell",
    5900:  "VNC — remote desktop, often left open",
    6666:  "IRC — used by botnets for command-and-control",
    6667:  "IRC — used by botnets for command-and-control",
    8080:  "HTTP alt — proxy/open-relay abuse",
    9001:  "Tor — anonymous routing",
    31337: "Back Orifice — classic backdoor trojan",
}

ENCRYPTED_PROTOCOLS = {
    "TLS", "TLSv1", "TLSv1.1", "TLSv1.2", "TLSv1.3",
    "SSL", "SSLv2", "SSLv3",
    "HTTPS", "SSH", "FTPS", "SMTPS", "IMAPS", "POP3S",
    "QUIC",
    "DTLS",
}
ROUTINE_PROTOCOLS = {
    "TCP", "UDP", "ICMP", "ICMPv6",
    "DNS", "MDNS", "LLMNR",
    "ARP", "NDP",
    "NTP", "SNMP",
    "HTTP",
    "DHCP", "DHCPv6",
    "SSDP", "IGMP", "IGMPv3",
}


def derive_features(src_ip, dst_ip, src_port, dst_port, protocol, pkt_size):

    proto_upper = protocol.strip().upper()

    size_feature = pkt_size

    bad_ports_found = []
    for p in [src_port, dst_port]:
        if p in SUSPICIOUS_PORTS:
            bad_ports_found.append((p, SUSPICIOUS_PORTS[p]))
    port_risk = len(bad_ports_found)   # 0, 1, or 2

    is_encrypted = any(proto_upper.startswith(ep.upper()) for ep in ENCRYPTED_PROTOCOLS)
    enc_score = 1 if is_encrypted else 0

    def is_private(ip):
        parts = ip.split(".")
        if len(parts) != 4:
            return False
        try:
            a, b = int(parts[0]), int(parts[1])
        except ValueError:
            return False
        return (a == 10 or
                a == 192 and b == 168 or
                a == 172 and 16 <= b <= 31 or
                a == 127)

    src_private = is_private(src_ip)
    dst_private = is_private(dst_ip)

    if src_private and dst_private:
        ip_risk = 0
    elif src_private or dst_private:
        ip_risk = 1
    else:
        ip_risk = 2

    explanation = {
        "encrypted":   is_encrypted,
        "bad_ports":   bad_ports_found,
        "src_private": src_private,
        "dst_private": dst_private,
        "ip_risk":     ip_risk,
        "protocol":    protocol,
    }

    return [size_feature, port_risk, enc_score, ip_risk], explanation

X_train = np.array([
    # size   port_risk  enc  ip_risk   Scenario
    [1200,      0,       1,    1   ],  # HTTPS to Google          -> Safe
    [ 980,      0,       1,    1   ],  # SSH session              -> Safe
    [ 850,      0,       0,    1   ],  # HTTP web page            -> Safe
    [1400,      0,       1,    1   ],  # Large TLS download       -> Safe
    [  66,      0,       0,    0   ],  # Internal LAN TCP ACK     -> Safe
    [ 200,      0,       0,    1   ],  # DNS query outbound       -> Safe
    [ 105,      0,       1,    1   ],  # Small TLS data           -> Safe
    [ 750,      0,       1,    1   ],  # Normal HTTPS session     -> Safe

    [  60,      1,       0,    1   ],  # Probe to SMB port 445    -> Suspicious
    [ 150,      1,       0,    2   ],  # External → RDP port 3389 -> Suspicious
    [4096,      1,       0,    1   ],  # Large pkt to FTP port 21 -> Suspicious
    [  80,      0,       0,    2   ],  # Unencrypted, both extern -> Suspicious
    [  50,      1,       1,    1   ],  # Encrypted but risky port -> Suspicious
    [ 512,      2,       0,    1   ],  # Two bad ports            -> Suspicious
    [ 100,      1,       0,    2   ],  # External to Telnet 23    -> Suspicious
    [2048,      1,       0,    1   ],  # Large pkt, one bad port  -> Suspicious

    [  40,      2,       0,    2   ],  # External, both bad ports -> Malicious
    [  28,      2,       0,    2   ],  # Tiny flood, 2 bad ports  -> Malicious
    [9000,      2,       0,    2   ],  # Huge pkt, 2 bad ports    -> Malicious
    [  40,      2,       0,    1   ],  # Backdoor port 4444       -> Malicious
    [  60,      2,       0,    2   ],  # Two bad ports, external  -> Malicious
    [4500,      1,       0,    2   ],  # Exfiltration via FTP     -> Malicious
    [  44,      2,       0,    2   ],  # NetBIOS exploit attempt  -> Malicious
    [  28,      2,       0,    2   ],  # IRC botnet C2 channel    -> Malicious
])

y_train = np.array([
    0, 0, 0, 0, 0, 0, 0, 0,   # Safe
    1, 1, 1, 1, 1, 1, 1, 1,   # Suspicious
    2, 2, 2, 2, 2, 2, 2, 2    # Malicious
])

LABEL_NAMES = {0: "SAFE", 1: "SUSPICIOUS", 2: "MALICIOUS"}
LABEL_ICONS = {0: "✓",   1: "!",          2: "✕"}

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

neural_net = MLPClassifier(
    hidden_layer_sizes=(12, 8, 6),
    activation='relu',
    max_iter=2000,
    random_state=42,
    learning_rate_init=0.01
)
neural_net.fit(X_scaled, y_train)

def analyze_packet(src_ip, dst_ip, src_port, dst_port,
                   protocol, pkt_size):


    features, expl = derive_features(
        src_ip, dst_ip, src_port, dst_port, protocol, pkt_size
    )

    x_scaled = scaler.transform([features])
    prediction    = neural_net.predict(x_scaled)[0]
    probabilities = neural_net.predict_proba(x_scaled)[0]

    label = LABEL_NAMES[prediction]
    icon  = LABEL_ICONS[prediction]

    enc_str  = "Yes (auto-detected)" if expl["encrypted"] else "No / unknown"
    priv_str = ("both private"
                if expl["src_private"] and expl["dst_private"]
                else "one external" if expl["src_private"] or expl["dst_private"]
                else "both external")
    print(f"     Encrypted (auto)  : {enc_str}")
    print(f"     IP scope  (auto)  : {priv_str}")

    if expl["bad_ports"]:
        for port, reason in expl["bad_ports"]:
            print(f"     Risky port (auto) : {port} — {reason}")
    else:
        print(f"     Risky port (auto) : None detected")

    print(f"     P(Safe)={probabilities[0]*100:.0f}%  "
          f"P(Suspicious)={probabilities[1]*100:.0f}%  "
          f"P(Malicious)={probabilities[2]*100:.0f}%")
    print()

    if expl["bad_ports"] and expl["ip_risk"] == 2:
        print("  >> RULE FIRED: External IP hitting a known-dangerous port.")
        print("     ACTION : Block source IP. Check firewall rules.")

    elif len(expl["bad_ports"]) == 2:
        print("  >> RULE FIRED: Both source AND destination ports are suspicious.")
        print("     ACTION : Very unusual — investigate this connection immediately.")

    elif expl["bad_ports"] and not expl["encrypted"]:
        port, reason = expl["bad_ports"][0]
        print(f"  >> RULE FIRED: Unencrypted traffic on risky port {port} ({reason}).")
        print("     ACTION : Flag for review. If unexpected, block.")

    elif pkt_size > 3000 and not expl["encrypted"] and expl["bad_ports"]:
        print("  >> RULE FIRED: Large unencrypted packet to suspicious port.")
        print("     ACTION : Possible data exfiltration — inspect and trace source.")

    elif expl["encrypted"] and not expl["bad_ports"] and expl["ip_risk"] <= 1:
        print("  >> OK: Encrypted traffic on standard ports — looks normal.")
        print("     ACTION : No action needed.")

    elif expl["ip_risk"] == 2 and not expl["encrypted"]:
        print("  >> NOTICE: Unencrypted traffic between two external IPs.")
        print("     ACTION : Unusual for a home/office network — review.")

    elif prediction == 1:
        print("  >> NOTICE: Some risk indicators present.")
        print("     ACTION : Monitor this source. Review if it recurs.")

    elif prediction == 2:
        print("  >> MALICIOUS: Multiple threat signals detected.")
        print("     ACTION : Block source IP and escalate.")

    else:
        print("  >> OK: No threat indicators found.")
        print("     ACTION : No action needed.")

    print()
    return label


try:
    src_ip   = input("  Source IP address       : ").strip() or "0.0.0.0"
    dst_ip   = input("  Destination IP address  : ").strip() or "0.0.0.0"

    src_port_str = input("  Source port (number)    : ").strip()
    dst_port_str = input("  Destination port (number): ").strip()
    src_port = int(src_port_str) if src_port_str.isdigit() else 0
    dst_port = int(dst_port_str) if dst_port_str.isdigit() else 0

    protocol = input("  Protocol (e.g. TCP, TLS, DNS): ").strip() or "TCP"

    size_str = input("  Packet size in bytes    : ").strip()
    pkt_size = int(size_str) if size_str.isdigit() else 66

    print()
    print(">>> Analysing your packet:")
    analyze_packet(src_ip, dst_ip, src_port, dst_port, protocol, pkt_size)

except (ValueError, KeyboardInterrupt):
    print("\n[Interactive mode skipped]")

print("=" * 60)
print("  Done.")
print("=" * 60)

  Source IP address       : 
  Destination IP address  : 
  Source port (number)    : 
  Destination port (number): 
  Protocol (e.g. TCP, TLS, DNS): 
  Packet size in bytes    : 

>>> Analysing your packet:
     Encrypted (auto)  : No / unknown
     IP scope  (auto)  : both external
     Risky port (auto) : None detected
     P(Safe)=0%  P(Suspicious)=100%  P(Malicious)=0%

  >> NOTICE: Unencrypted traffic between two external IPs.
     ACTION : Unusual for a home/office network — review.

  Done.
